# Report A & Report B — Competing Views

Two views over the same `radiology_revenue_cycle` data that produce **intentionally different KPIs** due to different business logic applied to ambiguous/defective data.

| KPI | Report A (CFO) | Report B (RevOps) | Root Cause |
|-----|---------------|-------------------|------------|
| Net Collection Rate | ~94% | ~82% | COMPLETENESS: NULL allowed handling |
| Denial Rate | ~6% | ~11% | VALIDITY: invalid CARC code inclusion |
| Days in AR | ~42 | ~51 | TIMELINESS: stale snapshot handling |
| Commercial Payer Mix | ~52% | ~38% | CONSISTENCY: payer name fragmentation |
| Total Charges | ~$47M | ~$49M | UNIQUENESS: rebill dedup + ACCURACY-A: TC/26 overlap |
| Net Revenue | ~$38M | ~$42M | ACCURACY-B/C: allowed_amount trust |

In [0]:
%sql
USE CATALOG serverless_stable_jc9zgx_catalog;
USE SCHEMA radiology_revenue_cycle;

## Report A: CFO Monthly Revenue Summary

**Philosophy**: "Only count what we can validate"
- Excludes NULL allowed_amount from net collections
- Drops invalid CARC codes from denial count
- Uses MAX(snapshot_date) per location for AR
- Joins to canonical payer table for payer mix
- Deduplicates rebills and removes TC/26 component overlaps
- Reconciles against payment table for revenue

In [0]:
%sql
CREATE OR REPLACE VIEW v_cfo_monthly_summary AS
WITH 
-- Deduplicate claims: keep latest claim_id per (patient, encounter, charge)
-- This removes the UNIQUENESS trap (unflagged rebills)
deduped_claims AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY encounter_id, patient_id, total_charge_amount
        ORDER BY filing_date DESC, claim_id DESC
      ) AS rn
    FROM claim_header
    WHERE claim_status != 'void'
  )
  WHERE rn = 1
),

-- Remove TC/26 component overlap lines (ACCURACY-A fix)
-- If a CPT appears with no modifier (global) AND with 26/TC on the same claim, keep only global
clean_lines AS (
  SELECT cl.*
  FROM claim_line cl
  JOIN deduped_claims dc ON cl.claim_id = dc.claim_id
  WHERE NOT EXISTS (
    -- Exclude component lines (26 or TC) where a global line exists for same CPT on same claim
    SELECT 1
    FROM claim_line gl
    WHERE gl.claim_id = cl.claim_id
      AND gl.cpt_code = cl.cpt_code
      AND gl.claim_line_id != cl.claim_line_id
      AND gl.modifier_1 IS NULL  -- global line exists
      AND cl.modifier_1 IN ('26', 'TC')  -- current line is a component
  )
),

-- Net Collections: exclude NULL allowed (COMPLETENESS fix)
net_collections AS (
  SELECT
    ROUND(SUM(paid_amount) / NULLIF(SUM(allowed_amount), 0) * 100, 1) AS net_collection_rate
  FROM clean_lines
  WHERE allowed_amount IS NOT NULL
    AND paid_amount IS NOT NULL
    AND paid_amount > 0
),

-- Denial Rate: only valid CARC codes (VALIDITY fix)
valid_denials AS (
  SELECT COUNT(DISTINCT d.claim_id) AS denied_claim_count
  FROM denial d
  JOIN deduped_claims dc ON d.claim_id = dc.claim_id
  WHERE d.carc_code NOT IN ('CO-999', 'PI-300', 'OA-500', 'PR-ZZZ')
),

-- Days in AR: use MAX snapshot_date per location (TIMELINESS fix — includes stale data)
ar_days AS (
  SELECT
    ROUND(
      SUM(outstanding_amount * 
        CASE aging_bucket
          WHEN 'current' THEN 15
          WHEN '31_60' THEN 45
          WHEN '61_90' THEN 75
          WHEN '91_120' THEN 105
          WHEN '121_plus' THEN 150
        END
      ) / NULLIF(SUM(outstanding_amount), 0)
    , 0) AS days_in_ar
  FROM (
    SELECT ar.*
    FROM ar_aging_snapshot ar
    INNER JOIN (
      SELECT location_id, MAX(snapshot_date) AS max_date
      FROM ar_aging_snapshot
      GROUP BY location_id
    ) latest ON ar.location_id = latest.location_id AND ar.snapshot_date = latest.max_date
  )
),

-- Commercial Mix: join to canonical payer table (CONSISTENCY fix)
payer_mix AS (
  SELECT
    ROUND(SUM(CASE WHEN p.payer_category = 'commercial' THEN dc.total_charge_amount ELSE 0 END) 
      / NULLIF(SUM(dc.total_charge_amount), 0) * 100, 1) AS commercial_pct
  FROM deduped_claims dc
  JOIN payer p ON dc.payer_id = p.payer_id
  WHERE dc.claim_status IN ('paid', 'partial', 'denied', 'pending')
),

-- Total Charges: deduped + TC/26 overlap removed
total_charges AS (
  SELECT ROUND(SUM(charge_amount) / 1000000, 1) AS total_charges_millions
  FROM clean_lines
),

-- Net Revenue: reconcile against payment table for PAY-011/012 (ACCURACY-B fix)
-- and use actual payments for BCBS Q1 2026 (ACCURACY-C fix)
net_revenue AS (
  SELECT ROUND(SUM(
    CASE 
      -- ACCURACY-B fix: for PAY-011/PAY-012, use payment amount instead of allowed
      WHEN dc.payer_id IN ('PAY-011', 'PAY-012') THEN COALESCE(pmt_agg.total_paid, cl.paid_amount, 0)
      -- ACCURACY-C fix: for BCBS Q1 2026, use payment amount where it exceeds allowed
      WHEN dc.payer_id = 'PAY-001' AND cl.service_date BETWEEN '2026-01-01' AND '2026-02-28'
        AND pmt_agg.total_paid > cl.allowed_amount * 1.05
        THEN pmt_agg.total_paid
      ELSE COALESCE(cl.paid_amount, 0)
    END
  ) / 1000000, 1) AS net_revenue_millions
  FROM clean_lines cl
  JOIN deduped_claims dc ON cl.claim_id = dc.claim_id
  LEFT JOIN (
    SELECT claim_line_id, SUM(payment_amount) AS total_paid
    FROM payment
    GROUP BY claim_line_id
  ) pmt_agg ON cl.claim_line_id = pmt_agg.claim_line_id
  WHERE dc.claim_status IN ('paid', 'partial')
)

SELECT
  'CFO Monthly Summary' AS report_name,
  'Finance' AS audience,
  nc.net_collection_rate,
  ROUND(vd.denied_claim_count * 100.0 / NULLIF((SELECT COUNT(DISTINCT claim_id) FROM deduped_claims WHERE claim_status != 'void'), 0), 1) AS denial_rate,
  ar.days_in_ar,
  pm.commercial_pct AS commercial_payer_mix,
  tc.total_charges_millions,
  nr.net_revenue_millions
FROM net_collections nc
CROSS JOIN valid_denials vd
CROSS JOIN ar_days ar
CROSS JOIN payer_mix pm
CROSS JOIN total_charges tc
CROSS JOIN net_revenue nr;

## Report B: Revenue Ops Performance Dashboard

**Philosophy**: "Count everything until proven invalid"
- Includes all claims (no dedup) — trusts claim_id as unique
- Substitutes charge_amount when allowed is NULL
- Includes ALL denials regardless of CARC validity
- Uses snapshot_date = CURRENT_DATE filter (excludes stale locations)
- Groups by claim_header.payer_name directly (no join to payer ref)
- Uses allowed_amount as-is from claim_line

In [0]:
%sql
CREATE OR REPLACE VIEW v_revops_performance AS
WITH 
-- No deduplication — trusts claim_id as unique (UNIQUENESS trap active)
-- No TC/26 overlap removal — sums all lines naively (ACCURACY-A trap active)
all_claims AS (
  SELECT *
  FROM claim_header
  WHERE claim_status != 'void'
),

-- Net Collections: substitute charge_amount when allowed is NULL (COMPLETENESS trap active)
net_collections AS (
  SELECT
    ROUND(
      SUM(cl.paid_amount) / 
      NULLIF(SUM(COALESCE(cl.allowed_amount, cl.charge_amount)), 0) * 100
    , 1) AS net_collection_rate
  FROM claim_line cl
  JOIN all_claims ac ON cl.claim_id = ac.claim_id
  WHERE cl.paid_amount IS NOT NULL
    AND cl.paid_amount > 0
),

-- Denial Rate: includes ALL denials, even invalid CARC codes (VALIDITY trap active)
all_denials AS (
  SELECT COUNT(DISTINCT d.claim_id) AS denied_claim_count
  FROM denial d
  JOIN all_claims ac ON d.claim_id = ac.claim_id
),

-- Days in AR: filter to CURRENT_DATE only (TIMELINESS trap active — excludes stale hospital locations)
ar_days AS (
  SELECT
    ROUND(
      SUM(outstanding_amount * 
        CASE aging_bucket
          WHEN 'current' THEN 15
          WHEN '31_60' THEN 45
          WHEN '61_90' THEN 75
          WHEN '91_120' THEN 105
          WHEN '121_plus' THEN 150
        END
      ) / NULLIF(SUM(outstanding_amount), 0)
    , 0) AS days_in_ar
  FROM ar_aging_snapshot
  WHERE snapshot_date = DATE '2026-05-15'  -- Only today's data (system_a locations excluded!)
),

-- Commercial Mix: naive pattern matching on claim_header.payer_name (CONSISTENCY trap active)
-- RevOps team doesn't maintain a canonical payer reference — uses ad-hoc string matching
-- MISSES: 'UHC', 'United HC', 'Blue Cross BS', 'BlueCross BlueShield of IL', 
--         'CVS/Aetna', 'AETNA HEALTH', 'Humana Inc', 'CIGNA HEALTH'
payer_mix AS (
  SELECT
    ROUND(
      SUM(CASE 
        WHEN payer_name LIKE '%Blue Cross Blue Shield%' 
          OR payer_name = 'BCBS'
          OR payer_name = 'BCBS IL'
          OR payer_name = 'United Healthcare'
          OR payer_name = 'UnitedHealth'
          OR payer_name = 'UNITED HEALTHCARE'
          OR payer_name = 'AETNA'
          OR payer_name = 'Aetna Inc'
          OR payer_name = 'CIGNA'
          OR payer_name = 'Cigna Health'
          OR payer_name = 'Humana'
          OR payer_name = 'HUMANA'
        THEN total_charge_amount
        ELSE 0
      END) / NULLIF(SUM(total_charge_amount), 0) * 100
    , 1) AS commercial_pct
  FROM all_claims
  WHERE claim_status IN ('paid', 'partial', 'denied', 'pending')
),

-- Total Charges: all lines, no dedup, no overlap removal (UNIQUENESS + ACCURACY-A traps active)
total_charges AS (
  SELECT ROUND(SUM(cl.charge_amount) / 1000000, 1) AS total_charges_millions
  FROM claim_line cl
  JOIN all_claims ac ON cl.claim_id = ac.claim_id
),

-- Net Revenue: uses allowed_amount as-is (ACCURACY-B/C traps active)
-- PAY-011/012 inflated allowed, BCBS stale rates — all trusted at face value
net_revenue AS (
  SELECT ROUND(SUM(
    COALESCE(cl.allowed_amount, cl.charge_amount) 
    - ABS(COALESCE(cl.adjustment_amount, 0))
  ) / 1000000, 1) AS net_revenue_millions
  FROM claim_line cl
  JOIN all_claims ac ON cl.claim_id = ac.claim_id
  WHERE ac.claim_status IN ('paid', 'partial')
)

SELECT
  'Revenue Ops Dashboard' AS report_name,
  'Operations' AS audience,
  nc.net_collection_rate,
  ROUND(ad.denied_claim_count * 100.0 / NULLIF((SELECT COUNT(DISTINCT claim_id) FROM all_claims), 0), 1) AS denial_rate,
  ar.days_in_ar,
  pm.commercial_pct AS commercial_payer_mix,
  tc.total_charges_millions,
  nr.net_revenue_millions
FROM net_collections nc
CROSS JOIN all_denials ad
CROSS JOIN ar_days ar
CROSS JOIN payer_mix pm
CROSS JOIN total_charges tc
CROSS JOIN net_revenue nr;

## Side-by-Side Comparison

In [0]:
%sql
CREATE OR REPLACE VIEW v_report_divergence AS
WITH a AS (SELECT * FROM v_cfo_monthly_summary),
     b AS (SELECT * FROM v_revops_performance)
SELECT
  'Net Collection Rate (%)' AS kpi,
  a.net_collection_rate AS report_a_cfo,
  b.net_collection_rate AS report_b_revops,
  ROUND(a.net_collection_rate - b.net_collection_rate, 1) AS divergence,
  'COMPLETENESS' AS root_cause,
  'NULL allowed_amount handling' AS explanation
FROM a, b
UNION ALL
SELECT
  'Denial Rate (%)',
  a.denial_rate,
  b.denial_rate,
  ROUND(a.denial_rate - b.denial_rate, 1),
  'VALIDITY',
  'Invalid CARC code inclusion'
FROM a, b
UNION ALL
SELECT
  'Days in AR',
  a.days_in_ar,
  b.days_in_ar,
  ROUND(a.days_in_ar - b.days_in_ar, 0),
  'TIMELINESS',
  'Stale snapshot handling'
FROM a, b
UNION ALL
SELECT
  'Commercial Payer Mix (%)',
  a.commercial_payer_mix,
  b.commercial_payer_mix,
  ROUND(a.commercial_payer_mix - b.commercial_payer_mix, 1),
  'CONSISTENCY',
  'Payer name fragmentation'
FROM a, b
UNION ALL
SELECT
  'Total Charges ($M)',
  a.total_charges_millions,
  b.total_charges_millions,
  ROUND(a.total_charges_millions - b.total_charges_millions, 1),
  'UNIQUENESS + ACCURACY',
  'Rebill dedup + TC/26 overlap'
FROM a, b
UNION ALL
SELECT
  'Net Revenue ($M)',
  a.net_revenue_millions,
  b.net_revenue_millions,
  ROUND(a.net_revenue_millions - b.net_revenue_millions, 1),
  'ACCURACY',
  'Allowed_amount trust (payer corrections)'
FROM a, b;

## Validate Report Divergence

In [0]:
%sql
SELECT * FROM v_cfo_monthly_summary;

In [0]:
%sql
SELECT * FROM v_revops_performance;

In [0]:
%sql
SELECT * FROM v_report_divergence;